In [2]:
import requests
import pandas as pd
import xml.etree.ElementTree as ET

In [3]:
SERVICE_KEY = "0d954a93b79fc80946f785697ee87efe1f220b8b979f44e8e9938942f9747b4d"

BASE_URL = "https://apis.data.go.kr/B551014/SRVC_TODZ_VDO_PKG"

In [4]:
# 6번 수집
url_fctr = BASE_URL + "/TODZ_VDO_ROUTINE_I"

all_data = []

for page in range(1, 4):
    params = {
        "serviceKey": SERVICE_KEY,
        "pageNo": page,
        "numOfRows": 1000,
        "resultType": "xml"
    }

    response = requests.get(url_fctr, params=params)
    response.raise_for_status()

    root = ET.fromstring(response.text)
    items = root.findall(".//item")

    for item in items:
        row = {}

        for child in item:
            row[child.tag] = child.text

        all_data.append(row)

df = pd.DataFrame(all_data)

In [5]:
def summarize_data(df):
    print(f"전체 행 수: {len(df):,}")
    print(f"전체 컬럼 수: {df.shape[1]}")
    print(f"고유 영상 수: {df['file_nm'].nunique():,}")
    
    print("\n체력요인 종류:")
    print(df["ftns_fctr_nm"].value_counts(dropna=False))

    print("\n중복 제거 후 체력요인별 구성 운동 수:")
    print(
        df[["file_nm", "trng_nm", "ftns_fctr_nm"]]
        .dropna(subset=["trng_nm", "ftns_fctr_nm"])
        .drop_duplicates()
        .groupby("ftns_fctr_nm")
        .size()
    )

In [6]:
summarize_data(df)

전체 행 수: 2,150
전체 컬럼 수: 27
고유 영상 수: 25

체력요인 종류:
ftns_fctr_nm
근력/근지구력    1043
유연성         622
민첩성/순발력     304
None        148
근력           33
Name: count, dtype: int64

중복 제거 후 체력요인별 구성 운동 수:
ftns_fctr_nm
근력          12
근력/근지구력    207
민첩성/순발력     57
유연성        184
dtype: int64


In [7]:
import math

# 4번 수집
url_mscl = BASE_URL + "/TODZ_VDO_MSCL_TRNG_I"

# 첫 요청으로 전체 데이터 개수 확인
params = {
    "serviceKey": SERVICE_KEY,
    "pageNo": 1,
    "numOfRows": 1000,
    "resultType": "xml"
}

response = requests.get(url_mscl, params=params)
response.raise_for_status()

root = ET.fromstring(response.text)

total_count = int(root.findtext(".//totalCount"))
total_pages = math.ceil(total_count / 1000)

# 전체 페이지 수집
all_mscl_data = []

for page in range(1, total_pages + 1):
    params["pageNo"] = page

    response = requests.get(url_mscl, params=params)
    response.raise_for_status()

    root = ET.fromstring(response.text)
    items = root.findall(".//item")

    for item in items:
        row = {}

        for child in item:
            row[child.tag] = child.text

        all_mscl_data.append(row)

df_mscl = pd.DataFrame(all_mscl_data)

In [8]:
# 7번 수집
url_all = BASE_URL + "/TODZ_VDO_VIEW_ALL_LIST_I"

# 첫 요청으로 전체 데이터 개수 확인
params = {
    "serviceKey": SERVICE_KEY,
    "pageNo": 1,
    "numOfRows": 1000,
    "resultType": "xml"
}

response = requests.get(url_all, params=params)
response.raise_for_status()

root = ET.fromstring(response.text)

total_count = int(root.findtext(".//totalCount"))
total_pages = math.ceil(total_count / 1000)

# 전체 페이지 수집
all_video_data = []

for page in range(1, total_pages + 1):
    params["pageNo"] = page

    response = requests.get(url_all, params=params)
    response.raise_for_status()

    root = ET.fromstring(response.text)
    items = root.findall(".//item")

    for item in items:
        row = {}

        for child in item:
            row[child.tag] = child.text

        all_video_data.append(row)

df_all = pd.DataFrame(all_video_data)

In [9]:
print("데이터 크기:", df_all.shape)
print("고유 영상 수:", df_all["file_nm"].nunique())

데이터 크기: (15045, 22)
고유 영상 수: 1083


In [10]:
# 1번 수집
url_guide = BASE_URL + "/TODZ_VDO_TRNG_GUIDE_I"

# 첫 요청으로 전체 데이터 개수 확인
params = {
    "serviceKey": SERVICE_KEY,
    "pageNo": 1,
    "numOfRows": 1000,
    "resultType": "xml"
}

response = requests.get(url_guide, params=params)
response.raise_for_status()

root = ET.fromstring(response.text)

total_count = int(root.findtext(".//totalCount"))
total_pages = math.ceil(total_count / 1000)

# 전체 페이지 수집
all_guide_data = []

for page in range(1, total_pages + 1):
    params["pageNo"] = page

    response = requests.get(url_guide, params=params)
    response.raise_for_status()

    root = ET.fromstring(response.text)
    items = root.findall(".//item")

    for item in items:
        row = {}

        for child in item:
            row[child.tag] = child.text

        all_guide_data.append(row)

df_guide = pd.DataFrame(all_guide_data)

In [11]:
print("전체 데이터 수:", total_count)
print("데이터 크기:", df_guide.shape)

전체 데이터 수: 8303
데이터 크기: (8303, 35)


In [12]:
print("[연령대]")
print(df_guide["aggrp_nm"].value_counts(dropna=False))

print("\n[체력요인]")
print(df_guide["ftns_fctr_nm"].value_counts(dropna=False))

print("\n[체력수준]")
print(df_guide["ftns_lvl_nm"].value_counts(dropna=False))

[연령대]
aggrp_nm
청소년    3356
어르신    2368
공통     1755
유소년     824
Name: count, dtype: int64

[체력요인]
ftns_fctr_nm
유연성        3168
근력         1516
근력/근지구력    1195
협응성         573
순발력         494
심폐지구력       425
민첩성         416
평형성         274
민첩성/순발력     146
협응력          55
전신지구력        29
유산소           7
민첩성/성능        5
Name: count, dtype: int64

[체력수준]
ftns_lvl_nm
1~5         2007
3           1328
4~5         1087
1~2          984
2~5          940
3~5          814
중급           737
공통           233
초급            98
5             26
None          25
3~5, 1~5      21
1              3
Name: count, dtype: int64


In [13]:
print("1번 고유 영상 수:", df_guide["file_nm"].nunique())

videos_1 = set(df_guide["file_nm"].dropna())
videos_7 = set(df_all["file_nm"].dropna())

print("7번과 겹치는 영상:", len(videos_1 & videos_7))
print("7번에 없는 1번 영상:", len(videos_1 - videos_7))

1번 고유 영상 수: 661
7번과 겹치는 영상: 661
7번에 없는 1번 영상: 0


In [14]:
# 영상 관계 확인
videos_1 = set(df_guide["file_nm"].dropna())
videos_4 = set(df_mscl["file_nm"].dropna())
videos_6 = set(df["file_nm"].dropna())

print("[1번 ↔ 4번]")
print("겹치는 영상:", len(videos_1 & videos_4))

print("\n[1번 ↔ 6번]")
print("겹치는 영상:", len(videos_1 & videos_6))

print("\n[4번 ↔ 6번]")
print("겹치는 영상:", len(videos_4 & videos_6))

[1번 ↔ 4번]
겹치는 영상: 0

[1번 ↔ 6번]
겹치는 영상: 0

[4번 ↔ 6번]
겹치는 영상: 0


In [15]:
# 5번 수집
url_std = BASE_URL + "/TODZ_VDO_STD_FTNS_I"

# 첫 요청으로 전체 데이터 개수 확인
params = {
    "serviceKey": SERVICE_KEY,
    "pageNo": 1,
    "numOfRows": 1000,
    "resultType": "xml"
}

response = requests.get(url_std, params=params)
response.raise_for_status()

root = ET.fromstring(response.text)

total_count = int(root.findtext(".//totalCount"))
total_pages = math.ceil(total_count / 1000)

# 전체 페이지 수집
all_std_data = []

for page in range(1, total_pages + 1):
    params["pageNo"] = page

    response = requests.get(url_std, params=params)
    response.raise_for_status()

    root = ET.fromstring(response.text)
    items = root.findall(".//item")

    for item in items:
        row = {}

        for child in item:
            row[child.tag] = child.text

        all_std_data.append(row)

df_std = pd.DataFrame(all_std_data)

In [16]:
print("전체 데이터 수:", total_count)
print("데이터 크기:", df_std.shape)

전체 데이터 수: 1827
데이터 크기: (1827, 26)


In [17]:
videos_5 = set(df_std["file_nm"].dropna())
videos_7 = set(df_all["file_nm"].dropna())

print("5번 고유 영상 수:", len(videos_5))
print("7번과 겹치는 영상:", len(videos_5 & videos_7))
print("7번에 없는 5번 영상:", len(videos_5 - videos_7))

print("\n[운동 주차]")
print(df_std["trng_week_nm"].value_counts(dropna=False).to_string())

print("\n[운동 순서]")
print(df_std["trng_sqnc_nm"].value_counts(dropna=False).to_string())

5번 고유 영상 수: 8
7번과 겹치는 영상: 8
7번에 없는 5번 영상: 0

[운동 주차]
trng_week_nm
3주차    580
4주차    567
1주차    396
2주차    284

[운동 순서]
trng_sqnc_nm
정리 운동    674
준비 운동    618
본 운동     522
None      13


In [18]:
# 3번 수집
url_video = BASE_URL + "/TODZ_VDO_TRNG_VIDEO_I"

# 첫 요청으로 전체 데이터 개수 확인
params = {
    "serviceKey": SERVICE_KEY,
    "pageNo": 1,
    "numOfRows": 1000,
    "resultType": "xml"
}

response = requests.get(url_video, params=params)
response.raise_for_status()

root = ET.fromstring(response.text)

total_count = int(root.findtext(".//totalCount"))
total_pages = math.ceil(total_count / 1000)

# 전체 페이지 수집
all_video_data_3 = []

for page in range(1, total_pages + 1):
    params["pageNo"] = page

    response = requests.get(url_video, params=params)
    response.raise_for_status()

    root = ET.fromstring(response.text)
    items = root.findall(".//item")

    for item in items:
        row = {}

        for child in item:
            row[child.tag] = child.text

        all_video_data_3.append(row)

df_video = pd.DataFrame(all_video_data_3)

In [19]:
print("전체 데이터 수:", total_count)
print("데이터 크기:", df_video.shape)
print("\n컬럼:")
print(df_video.columns.tolist())

전체 데이터 수: 1668
데이터 크기: (1668, 32)

컬럼:
['trng_mscl_nm', 'rptt_tcnt_nm', 'file_url', 'vdo_desc', 'file_sz', 'fps_cnt', 'row_num', 'resolution', 'tool_nm', 'aggrp_nm', 'frme_no', 'ecrg_cycl_nm', 'img_file_nm', 'trng_mscl_zn_nm', 'fbctn_yr', 'trng_plc_nm', 'vdo_len', 'trng_mscl_part', 'lang', 'trng_nm', 'job_ymd', 'trng_mscl_class', 'vdo_ttl_nm', 'snap_tm', 'trng_hr_nm', 'file_type_nm', 'file_nm', 'trng_mscl_eng_nm', 'img_file_url', 'img_file_sn', 'data_type', 'set_cnt_nm']


In [20]:
videos_3 = set(df_video["file_nm"].dropna())

print("3번 고유 영상 수:", len(videos_3))
print("7번과 겹치는 영상:", len(videos_3 & videos_7))
print("7번에 없는 3번 영상:", len(videos_3 - videos_7))

print("\n[다른 API와 중복]")
print("1번과 중복:", len(videos_3 & videos_1))
print("4번과 중복:", len(videos_3 & videos_4))
print("5번과 중복:", len(videos_3 & videos_5))
print("6번과 중복:", len(videos_3 & videos_6))

3번 고유 영상 수: 243
7번과 겹치는 영상: 243
7번에 없는 3번 영상: 0

[다른 API와 중복]
1번과 중복: 0
4번과 중복: 0
5번과 중복: 0
6번과 중복: 0


                     [7번 전체 동영상]
                        1,083개
                           │
       ┌────────┬──────────┼─────────┬─────────┐
       ↓        ↓          ↓         ↓         ↓
      1번       3번        4번        5번       6번
     661개     243개      114개      8개       25개
   처방가이드   처방영상     근골격계     표준운동    목적루틴

     +
   미분류 32개

In [ ]:
# 데이터 수집 완료
import os

os.makedirs("data/raw", exist_ok=True)

df_guide.to_csv("data/raw/api_1_guide.csv", index=False, encoding="utf-8-sig")
df_video.to_csv("data/raw/api_3_video.csv", index=False, encoding="utf-8-sig")
df_mscl.to_csv("data/raw/api_4_mscl.csv", index=False, encoding="utf-8-sig")
df_std.to_csv("data/raw/api_5_std.csv", index=False, encoding="utf-8-sig")
df.to_csv("data/raw/api_6_routine.csv", index=False, encoding="utf-8-sig")
df_all.to_csv("data/raw/api_7_all.csv", index=False, encoding="utf-8-sig")